# Phase 1: 风控数据基础

**学习目标**
- 理解风控数据的核心表结构（申请/借据/还款 三张表）
- 掌握观察期 vs 表现期的概念
- 学会用SQL从风控数仓取数

**真实公司里的角色**
- 风控分析师从数仓团队拿数据
- 你写的每一句SQL都会被评审
- 取数逻辑错误 = 模型结果全部作废

**面试常问**
- '风控数据仓库有哪些核心表？分别记录什么信息？'
- '什么是观察期和表现期？为什么需要这两个窗口？'

## 1. 风控数仓的3张核心表

在银行/互联网金融公司，风控数仓以这三张表为核心：

```
申请时                    放款后                         还款中
apply_table ──────────> loan_table ─────────────> repayment_table
 (申请表)      审批通过     (借据表)      每期还款      (还款表)
 1人可多笔                1笔申请=1笔借据             1笔借据=N期还款
```

**关键区别**：
- apply_table 存的是"申请时刻的快照"（申请时的FICO分、收入等）
- loan_table 存的是"放款后的信息"（实际利率、还款状态等）
- repayment_table 存的是"每一期的还款明细"（是否逾期、逾期几天）

> 面试技巧: 问到"风控数据库怎么设计"时，画出这三张表的ER图就是满分答案。

In [ ]:
import sqlite3
import pandas as pd
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
sys.path.append('..')

from config import DB_PATH

conn = sqlite3.connect(DB_PATH)

# 查看表结构
for table in ['apply_table', 'loan_table', 'repayment_table']:
    print(f'\n=== {table} ===')
    schema = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    display(schema[['name', 'type']])

## 2. 理解时间窗口：风控最核心的概念

### 为什么需要时间窗口？

你不能用"未来"的信息去预测"过去"——这叫**数据穿越(Data Leakage)**，是风控建模最严重的错误。

```
时间轴 ─────────────────────────────────────────────────>

  |<---- 观察期 (24个月) ---->|<---- 表现期 (12个月) ---->|
  |                           |                          |
  观察点                      放款日                    表现点
 (取历史特征)                (申请/放款)               (看是否逾期)
```

- **观察期(Observation Window)**: 看借款人历史行为的窗口，通常是借款前12-24个月
  - 例：近6个月查询次数、近2年逾期次数、当前FICO分
- **表现期(Performance Window)**: 看借款后还款表现的窗口，通常6-12个月
  - 例：放款后12个月内是否逾期90天以上

> 面试必问: '怎么定义好坏客户？'
> 标准答案: '表现期内，逾期>=90天(M3+)的为坏客户；逾期<30天的为好客户。中间客户(M1-M2)剔除，因为他们的好坏不明确，留着会污染模型。'

In [ ]:
# 验证: 查看借据的放款时间范围
dates = pd.read_sql_query('''
    SELECT 
        MIN(issue_date) as earliest_loan,
        MAX(issue_date) as latest_loan,
        COUNT(*) as total_loans
    FROM loan_table
''', conn)
display(dates)
print(f"放款时间跨度: {dates['earliest_loan'][0]} 到 {dates['latest_loan'][0]}")
print("表现期需要到放款后12个月才能判断好坏 -> 所以能用于建模的只有早期放款的客户")

## 3. 关键风控SQL查询

在真实工作里，风控分析师大量时间在写SQL。以下是最常用的几个查询。

In [ ]:
# ---- 查询1: 整体资产质量概览 ----
# 面试中: '给你一个借贷数据集，你先看什么？'
# 答案: 先看整体逾期率和坏账率，判断资产质量基准线

overview = pd.read_sql_query('''
    SELECT 
        loan_status,
        COUNT(*) as cnt,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
        ROUND(AVG(loan_amnt), 0) as avg_loan_amnt,
        ROUND(AVG(int_rate), 2) as avg_int_rate,
        ROUND(AVG(fico_score), 0) as avg_fico
    FROM loan_table
    GROUP BY loan_status
''', conn)
display(overview)
print(f"\n坏账率: {overview[overview['loan_status']=='Bad']['pct'].values[0]:.1f}%")

In [ ]:
# ---- 查询2: 不同借款目的的坏账率 ----
# 面试中: '哪些借款目的风险最高？'

purpose_risk = pd.read_sql_query('''
    SELECT 
        a.purpose,
        COUNT(*) as total,
        SUM(CASE WHEN l.loan_status = 'Bad' THEN 1 ELSE 0 END) as bad_cnt,
        ROUND(SUM(CASE WHEN l.loan_status = 'Bad' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as bad_rate,
        ROUND(AVG(l.loan_amnt), 0) as avg_loan_amnt
    FROM loan_table l
    JOIN apply_table a ON l.application_id = a.application_id
    GROUP BY a.purpose
    ORDER BY bad_rate DESC
''', conn)
display(purpose_risk)

In [ ]:
# ---- 查询3: FICO分段的坏账率 ----
# FICO分是最重要的信用风险指标
# 面试中: 'FICO分为什么能预测违约？'
# 答案: FICO综合了还款历史、负债水平、信用长度等5大维度，是个人信用风险的浓缩指标

fico_risk = pd.read_sql_query('''
    SELECT 
        CASE 
            WHEN fico_score < 620 THEN '<620 (差)'
            WHEN fico_score < 660 THEN '620-660 (一般)'
            WHEN fico_score < 700 THEN '660-700 (良好)'
            WHEN fico_score < 740 THEN '700-740 (很好)'
            ELSE '740+ (优秀)'
        END as fico_band,
        COUNT(*) as cnt,
        ROUND(SUM(CASE WHEN loan_status = 'Bad' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as bad_rate
    FROM loan_table
    GROUP BY fico_band
    ORDER BY MIN(fico_score)
''', conn)
display(fico_risk)

In [ ]:
# ---- 查询4: 还款逾期分布 ----
# 从还款表看: 有多少笔还款是逾期的?

repay_stats = pd.read_sql_query('''
    SELECT 
        dpd,
        CASE 
            WHEN dpd = 0 THEN '按时还款'
            WHEN dpd < 30 THEN '逾期1-29天'
            WHEN dpd < 60 THEN 'M1 (30-59天)'
            WHEN dpd < 90 THEN 'M2 (60-89天)'
            ELSE 'M3+ (90天以上)'
        END as dpd_label,
        COUNT(*) as cnt
    FROM repayment_table
    GROUP BY dpd_label
    ORDER BY dpd
''', conn)
display(repay_stats)

In [ ]:
conn.close()

## Phase 1 总结

### 你学会了什么
1. 风控数仓的三张核心表: apply_table(快照) / loan_table(借据) / repayment_table(明细)
2. 时间窗口概念: 观察期(取特征) vs 表现期(看标签) —— 这是风控最最基础的概念
3. 基本的风险分析SQL: 整体坏账率、分维度坏账率、逾期分布

### 简历可写
> 设计并搭建了风控数据仓库模型（申请-借据-还款三层架构），
> 使用SQL完成5万笔借贷数据的风险画像初步分析，
> 掌握观察期/表现期时间窗口划分方法，为后续评分卡开发奠定数据基础。

### 下一阶段
Phase 2: EDA与风险画像 — 深入分析谁在违约、为什么违约